# Clinical Transformer

For installation and dependencies, refer to the original publication (PMID: 40025003) and https://codeocean.com/capsule/2256163/tree/v1

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import os
import pickle
import random
import time

PROJECT_ROOT = Path('/home/boscoll/Projects/cdk_predict')
TRITON_CACHE = Path('/tmp/clinical-transformer-triton-cache')
TRITON_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('TRITON_CACHE_DIR', str(TRITON_CACHE))

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold, train_test_split
from sksurv.metrics import concordance_index_censored
from transformers import BertConfig
from clinical_transformer import vnBertPretrainedModel, vnBertTokenizerTabular

SEED = 94
DEVICE = 'cuda'
INNER_VALIDATION_FRACTION = 0.20
MAX_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 40
MIN_DELTA = 1e-4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0
CINDEX_SIGMA = 1.0
HIDDEN_SIZE = 64
NUM_HIDDEN_LAYERS = 2
NUM_ATTENTION_HEADS = 4
INTERMEDIATE_SIZE = 128
DROPOUT = 0.10
MASK_PROBABILITY = 0.20
PRETRAINING_STEPS = 20_000
PRETRAINING_BATCH_SIZE = 128
PRETRAINING_LEARNING_RATE = 1e-3
VALUE_LOSS_WEIGHT = 1.0

OOF_RISK_PATH = PROJECT_ROOT / 'results/model_benchmark_aug2026/clinical_transformer_risks.csv'
OUTPUT_COLUMNS = [
    'clinicogenomic__clinical_transformer_direct',
    'clinicogenomic__clinical_transformer_gradual',
]

print(f'clinical-transformer: {metadata.version("clinical-transformer")}')
print(f'CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not visible. Select Python (transformer_env) and restart the kernel.')
torch.cuda.set_device(0)
print(f'GPU: {torch.cuda.get_device_name(0)}')

/home/boscoll/miniconda3/envs/transformer_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/boscoll/miniconda3/envs/transformer_env/bin/x86_64-conda-linux-gnu-ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/boscoll/miniconda3/envs/transformer_env/bin/x86_64-conda-linux-gnu-ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `dlvsym'
/home/boscoll/miniconda3/envs/transformer_env/bin/x86_64-conda-linux-gnu-ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `dladdr'
/home/boscoll/miniconda3/envs/transformer_env/bin/x86_64-conda-linux-gnu-ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `dlopen'
/home/boscoll/miniconda3/envs/transformer_env/bin/x86_64-conda-linux-gnu-ld: /usr/local/cuda/lib64/libcufile.so: undefin

clinical-transformer: 2.0.8
CUDA available: True
GPU: NVIDIA A40


## Import

In [ ]:
project_path = Path('/home/boscoll/Projects/cdk_predict')
train_test_split = project_path / 'data/development_cohort/train_test_split/genomic_train_test_split_msk_final'
clinical_path = project_path / 'data/development_cohort/clinical/clinical_msk_final.csv'

with open(train_test_split, 'rb') as f:
    X_train_genomic, X_holdout_locked, y_train, y_holdout_locked = pickle.load(f)
y_train = y_train.loc[X_train_genomic.index, ['Time', 'Event']].copy()
clinical = pd.read_csv(clinical_path, index_col=0)

X_train = pd.concat([clinical.loc[X_train_genomic.index], X_train_genomic], axis=1) 
categorical_features = [column for column in X_train.columns]

## Tokenizer, model, and training helpers

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def survival_strata(y):
    time_quartile = pd.qcut(y['Time'].rank(method='first'), 4, labels=False)
    return y['Event'].astype(str) + '_' + time_quartile.astype(str)


def dataframe_records(frame):
    return frame.replace([np.inf, -np.inf], np.nan).to_dict(orient='records')


def fit_tokenizer(frame):
    tokenizer = vnBertTokenizerTabular(
        categorical_features=categorical_features,
    )
    tokenizer.fit(
        dataframe_records(frame),
        categorical_features=categorical_features,
    )
    return tokenizer


def encode_frame(tokenizer, frame):
    encoded = tokenizer(
        dataframe_records(frame),
        return_minmax_values=True,
        return_attention_mask=True,
    )
    max_length = len(frame.columns) + 1
    token_rows, value_rows = [], []
    for ids, values in zip(encoded['input_ids'], encoded['minmax_values']):
        ids = [tokenizer.cls_token_id] + list(ids)
        values = [1.0] + list(values)
        padding = max_length - len(ids)
        if padding < 0:
            raise ValueError('Encoded sequence is longer than the feature vocabulary.')
        token_rows.append(ids + [tokenizer.pad_token_id] * padding)
        value_rows.append(values + [0.0] * padding)
    return (
        torch.tensor(token_rows, dtype=torch.long),
        torch.tensor(value_rows, dtype=torch.float32),
    )


def survival_tensor(y):
    return torch.tensor(y[['Time', 'Event']].to_numpy(), dtype=torch.float32)

class DirectClinicalTransformer(torch.nn.Module):
    def __init__(self, tokenizer):
        super().__init__()
        self.model_config = BertConfig(
            vocab_size=tokenizer.vocab_size,
            hidden_size=HIDDEN_SIZE,
            num_hidden_layers=NUM_HIDDEN_LAYERS,
            num_attention_heads=NUM_ATTENTION_HEADS,
            intermediate_size=INTERMEDIATE_SIZE,
            hidden_dropout_prob=DROPOUT,
            attention_probs_dropout_prob=DROPOUT,
            pad_token_id=tokenizer.pad_token_id,
        )
        self.encoder = vnBertPretrainedModel(self.model_config)
        self.risk_head = torch.nn.Linear(HIDDEN_SIZE, 1, bias=False)
        torch.nn.init.xavier_uniform_(self.risk_head.weight)

    def encode(self, tokens, values):
        padding_mask = tokens.ne(self.model_config.pad_token_id)
        masked_positions = values.eq(-10.0)
        key_value_mask = padding_mask & ~masked_positions
        values_for_embedding = torch.where(masked_positions, 0.0, values)
        hidden_state, _ = self.encoder.embedder(tokens=tokens, values=values_for_embedding)
        padding_expanded = padding_mask.unsqueeze(-1).to(hidden_state.dtype)
        hidden_state = hidden_state * padding_expanded

        batch_size, sequence_length = tokens.shape
        attention_mask = padding_mask[:, None, None, :].expand(
            batch_size, 1, sequence_length, sequence_length
        )
        attention_mask = attention_mask & key_value_mask[:, None, None, :]
        diagonal = torch.eye(sequence_length, device=tokens.device, dtype=torch.bool)
        masked_self = masked_positions[:, None, :] & diagonal[None, :, :]
        attention_mask = attention_mask | masked_self[:, None, :, :]
        attention_mask = (-1e6 * ~attention_mask).to(hidden_state.dtype)

        for layer in self.encoder.encoder:
            layer_output = layer(
                hidden_states=hidden_state,
                attention_mask=attention_mask,
                output_attentions=False,
            )
            hidden_state = (
                layer_output[0] if isinstance(layer_output, (tuple, list)) else layer_output
            )
            hidden_state = hidden_state * padding_expanded

        return self.encoder.output_ln(hidden_state) *padding_expanded

    def forward(self, tokens, values):
        return self.risk_head(self.encode(tokens, values)[:, 0, :]).squeeze(-1)


def smooth_cindex_loss(labels, risk_score):
    times = labels[:, 0]
    events = labels[:, 1].bool()
    comparable = events[:, None] & (times[:, None] < times[None, :])
    if not comparable.any():
        raise RuntimeError('No comparable survival pairs.')
    risk_difference = risk_score[:, None] - risk_score[None, :]
    return 1.0 - torch.sigmoid(risk_difference / CINDEX_SIGMA)[comparable].mean()


@torch.no_grad()
def predict_risk(model, tokens, values, batch_size=256):
    model.eval()
    predictions = []
    for start in range(0, len(tokens), batch_size):
        predictions.append(
            model(
                tokens[start:start + batch_size].to(DEVICE),
                values[start:start + batch_size].to(DEVICE),
            ).cpu()
        )
    return torch.cat(predictions).numpy()


def harrell_cindex(y, risk):
    return concordance_index_censored(
        y['Event'].astype(bool), y['Time'].astype(float), np.asarray(risk)
    )[0]


def train_epoch(model, optimizer, tokens, values, labels):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    risk = model(tokens.to(DEVICE), values.to(DEVICE))
    loss = smooth_cindex_loss(labels.to(DEVICE), risk)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
    optimizer.step()
    return float(loss.detach().cpu())

In [4]:
class MaskedFeaturePretrainer(torch.nn.Module):
    def __init__(self, tokenizer):
        super().__init__()
        self.backbone = DirectClinicalTransformer(tokenizer)
        self.token_head = torch.nn.Linear(HIDDEN_SIZE, tokenizer.vocab_size)
        self.value_head = torch.nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, tokens, values):
        hidden = self.backbone.encode(tokens, values)
        return self.token_head(hidden), self.value_head(hidden).squeeze(-1)


def make_masked_batch(tokens, values, tokenizer):
    positions = torch.randint(
        len(tokens), (min(PRETRAINING_BATCH_SIZE, len(tokens)),)
    )
    batch_tokens = tokens[positions].to(DEVICE)
    batch_values = values[positions].to(DEVICE)
    eligible = (
        batch_tokens.ne(tokenizer.pad_token_id)
        & batch_tokens.ne(tokenizer.cls_token_id)
    )
    mask = torch.rand(batch_tokens.shape, device=DEVICE).lt(MASK_PROBABILITY) & eligible
    for row in torch.where(~mask.any(dim=1))[0]:
        candidates = torch.where(eligible[row])[0]
        mask[row, candidates[torch.randint(len(candidates), (1,), device=DEVICE)]] = True

    corrupted_tokens = batch_tokens.clone()
    corrupted_tokens[mask] = tokenizer.mask_token_id
    return corrupted_tokens, batch_values, batch_tokens, mask


def pretrain_encoder(tokenizer, tokens, values, seed):
    set_seed(seed)
    model = MaskedFeaturePretrainer(tokenizer).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=PRETRAINING_LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    started = time.perf_counter()

    for step in range(1, PRETRAINING_STEPS + 1):
        corrupted, visible_values, original_tokens, mask = make_masked_batch(
            tokens, values, tokenizer
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        token_logits, predicted_values = model(corrupted, visible_values)
        token_loss = torch.nn.functional.cross_entropy(
            token_logits[mask], original_tokens[mask]
        )
        value_loss = torch.nn.functional.mse_loss(
            predicted_values[mask], visible_values[mask]
        )
        loss = token_loss + VALUE_LOSS_WEIGHT * value_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
        optimizer.step()
        if step == 1 or step % 5_000 == 0:
            print(f'  pretraining step {step}/{PRETRAINING_STEPS}: loss={loss.item():.4f}')

    encoder_state = {
        key: value.detach().cpu().clone()
        for key, value in model.backbone.encoder.state_dict().items()
    }
    seconds = time.perf_counter() - started
    del model
    torch.cuda.empty_cache()
    return encoder_state, seconds

## Nested epoch selection

In [ ]:
#epoch selection, separatedly for direct and gradual learning, with each limited to the training partitions

def select_epoch(X_outer_train, y_outer_train, seed, gradual=False):
    inner_train_index, inner_validation_index = train_test_split(
        X_outer_train.index,
        test_size=INNER_VALIDATION_FRACTION,
        random_state=seed,
        stratify=survival_strata(y_outer_train),
    )
    tokenizer = fit_tokenizer(X_outer_train.loc[inner_train_index])
    train_tokens, train_values = encode_frame(tokenizer, X_outer_train.loc[inner_train_index])
    validation_tokens, validation_values = encode_frame(
        tokenizer, X_outer_train.loc[inner_validation_index]
    )
    train_labels = survival_tensor(y_outer_train.loc[inner_train_index])

    encoder_state = None
    pretraining_seconds = 0.0
    if gradual:
        encoder_state, pretraining_seconds = pretrain_encoder(
            tokenizer, train_tokens, train_values, seed
        )

    set_seed(seed)
    model = DirectClinicalTransformer(tokenizer).to(DEVICE)
    if encoder_state is not None:
        model.encoder.load_state_dict(encoder_state)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    best_epoch = 1
    best_cindex = -np.inf
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        loss = train_epoch(
            model, optimizer, train_tokens, train_values, train_labels
        )
        validation_risk = predict_risk(model, validation_tokens, validation_values)
        validation_cindex = harrell_cindex(
            y_outer_train.loc[inner_validation_index], validation_risk
        )
        history.append({
            'epoch': epoch, 'loss': loss, 'validation_cindex': validation_cindex
        })

        if validation_cindex > best_cindex + MIN_DELTA:
            best_epoch = epoch
            best_cindex = validation_cindex
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            break

    del model
    torch.cuda.empty_cache()
    return best_epoch, best_cindex, pd.DataFrame(history), pretraining_seconds

In [10]:
def fit_outer_fold(
    X_outer_train, y_outer_train, X_outer_validation, epochs, seed, gradual=False
):
    tokenizer = fit_tokenizer(X_outer_train)
    train_tokens, train_values = encode_frame(tokenizer, X_outer_train)
    validation_tokens, validation_values = encode_frame(tokenizer, X_outer_validation)
    train_labels = survival_tensor(y_outer_train)

    encoder_state = None
    pretraining_seconds = 0.0
    if gradual:
        encoder_state, pretraining_seconds = pretrain_encoder(
            tokenizer, train_tokens, train_values, seed
        )

    set_seed(seed)
    model = DirectClinicalTransformer(tokenizer).to(DEVICE)
    if encoder_state is not None:
        model.encoder.load_state_dict(encoder_state)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    for _ in range(epochs):
        train_epoch(model, optimizer, train_tokens, train_values, train_labels)

    train_risk = predict_risk(model, train_tokens, train_values)
    validation_risk = predict_risk(model, validation_tokens, validation_values)
    lower, upper = np.quantile(train_risk, [0.01, 0.99])
    if upper <= lower:
        raise RuntimeError('Outer-training risk is constant and cannot be normalized.')
    validation_risk = np.clip(
        (np.clip(validation_risk, lower, upper) - lower) / (upper - lower) * 100,
        0, 100,
    )

    del model
    torch.cuda.empty_cache()
    return validation_risk, pretraining_seconds

## Generate five-fold OOF risks

In [ ]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
outer_splits = list(outer_cv.split(X_train, survival_strata(y_train)))
oof_risk = pd.DataFrame(np.nan, index=X_train.index, columns=OUTPUT_COLUMNS)
fold_rows = []
selection_histories = {}
arms = {
    'direct': False,
    'gradual': True,
}


In [ ]:

for fold, (train_position, validation_position) in enumerate(outer_splits, start=1):
    train_index = X_train.index[train_position]
    validation_index = X_train.index[validation_position]
    fold_seed = SEED + fold
    for arm, gradual in arms.items():
        started = time.perf_counter()
        output_column = f'clinicogenomic__clinical_transformer_{arm}'
        print(f'Fold {fold}/{5}, {arm}')
        best_epoch, inner_cindex, history, inner_pretraining_seconds = select_epoch(
            X_train.loc[train_index],
            y_train.loc[train_index],
            fold_seed,
            gradual=gradual,
        )
        selection_histories[(fold, arm)] = history
        validation_risk, outer_pretraining_seconds = fit_outer_fold(
            X_train.loc[train_index],
            y_train.loc[train_index],
            X_train.loc[validation_index],
            best_epoch,
            fold_seed,
            gradual=gradual,
        )
        oof_risk.loc[validation_index, output_column] = validation_risk
        fold_rows.append({
            'fold': fold,
            'arm': arm,
            'n_outer_train': len(train_index),
            'n_outer_validation': len(validation_index),
            'selected_epoch': best_epoch,
            'inner_validation_cindex': inner_cindex,
            'inner_pretraining_seconds': inner_pretraining_seconds,
            'outer_pretraining_seconds': outer_pretraining_seconds,
            'total_seconds': time.perf_counter() - started,
        })
        print(
            f'  epoch={best_epoch}, inner C-index={inner_cindex:.3f}, '
            f'{fold_rows[-1]["total_seconds"]:.1f}s'
        )

assert oof_risk.notna().all().all(), 'Some development patients did not receive both OOF risks'
fold_audit = pd.DataFrame(fold_rows)
display(fold_audit)
for column in OUTPUT_COLUMNS:
    print(f'{column}: pooled OOF C-index={harrell_cindex(y_train, oof_risk[column]):.3f}')

Fold 1/5, direct


Fitting tokenizer (pass 1): 100%|██████████| 565/565 [00:00<00:00, 25979.61it/s]
INFO	2026-08-28 15:23:17,142	Fit complete – vocab_size=65, categorical=2, numerical=59
Tokenizing samples:   0%|          | 0/565 [00:00<?, ?it/s]

Fitting tokenizer (pass 1): 100%|██████████| 707/707 [00:00<00:00, 25924.49it/s]
INFO	2026-08-28 15:23:20,221	Fit complete – vocab_size=65, categorical=2, numerical=59
Tokenizing samples: 100%|██████████| 177/177 [00:00<00:00, 18439.94it/s]


  epoch=216, inner C-index=0.698, 4.7s
Fold 1/5, gradual


Fitting tokenizer (pass 1): 100%|██████████| 565/565 [00:00<00:00, 25768.02it/s]
INFO	2026-08-28 15:23:21,854	Fit complete – vocab_size=65, categorical=2, numerical=59
Tokenizing samples: 100%|██████████| 142/142 [00:00<00:00, 18701.05it/s]

...

  pretraining step 5000/20000: loss=2.6449
  pretraining step 10000/20000: loss=2.5443
  pretraining step 15000/20000: loss=2.5281
  pretraining step 20000/20000: loss=2.5370


Fitting tokenizer (pass 1): 100%|██████████| 708/708 [00:00<00:00, 26188.05it/s]
INFO	2026-08-28 15:37:46,286	Fit complete – vocab_size=65, categorical=2, numerical=59
Tokenizing samples: 100%|██████████| 176/176 [00:00<00:00, 18443.41it/s]


  pretraining step 1/20000: loss=4.7953
  pretraining step 5000/20000: loss=2.6696
  pretraining step 10000/20000: loss=2.4918
  pretraining step 15000/20000: loss=2.5143
  pretraining step 20000/20000: loss=2.5332
  epoch=75, inner C-index=0.621, 180.0s


,fold,arm,n_outer_train,n_outer_validation,selected_epoch,inner_validation_cindex,inner_pretraining_seconds,outer_pretraining_seconds,total_seconds
0,1,direct,707,177,216,0.698033,0.000000,0.000000,4.712405
1,1,gradual,707,177,105,0.692467,114.649790,90.045955,207.012765
2,2,direct,707,177,175,0.687686,0.000000,0.000000,3.357907
3,2,gradual,707,177,172,0.690535,89.931871,89.074593,182.357552
4,3,direct,707,177,153,0.672692,0.000000,0.000000,3.024985
5,3,gradual,707,177,211,0.665641,89.194688,91.328208,184.475693
6,4,direct,707,177,117,0.693833,0.000000,0.000000,2.449195
7,4,gradual,707,177,298,0.690379,93.800295,90.391739,189.182885
8,5,direct,708,176,98,0.592030,0.000000,0.000000,2.158203
9,5,gradual,708,176,75,0.620999,89.265957,88.961645,180.034789


clinicogenomic__clinical_transformer_direct: pooled OOF C-index=0.675
clinicogenomic__clinical_transformer_gradual: pooled OOF C-index=0.663


In [ ]:
# fold_audit.to_csv('/home/boscoll/Projects/cdk_predict/results/model_benchmark_aug2026/clinical_transformer_fold_audit.csv', index=False)

In [ ]:
oof_risk.index.name = 'record_id'
# oof_risk.to_csv(OOF_RISK_PATH)